# ConvNeXt CIFAR-100 — Colab T4 训练

In [ ]:
# 安装依赖
!pip install pyyaml tensorboard -q

In [ ]:
# 挂载 Google Drive 并解压项目文件
# 步骤: 先把 cifar100.zip 上传到 Google Drive 根目录（浏览器 drive.google.com 拖拽上传）
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

drive_zip = '/content/drive/MyDrive/cifar100.zip'
extract_dir = '/content/cifar100_project'

if os.path.exists(drive_zip):
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(drive_zip, 'r') as z:
        z.extractall(extract_dir)
    print(f'已解压到 {extract_dir}')
else:
    print(f'未找到 {drive_zip}')
    print('请确认已将 cifar100.zip 上传到 Google Drive 根目录')

# 检查解压后的目录结构
print('\n解压后的目录结构:')
for root, dirs, files in os.walk(extract_dir):
    level = root.replace(extract_dir, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        for f in files[:5]:
            print(f'{indent}  {f}')
        if len(files) > 5:
            print(f'{indent}  ... 共 {len(files)} 个文件')

In [ ]:
# 下载 CIFAR-100 数据集
import torchvision, os

dataset_root = '/content/cifar100'
os.makedirs(dataset_root, exist_ok=True)
torchvision.datasets.CIFAR100(root=dataset_root, train=True, download=True)
torchvision.datasets.CIFAR100(root=dataset_root, train=False, download=True)
print('CIFAR-100 下载完成')

In [ ]:
# 确认 GPU
import torch
print(f'CUDA 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# 训练 - 自动查找 train_convnext.py 所在目录
import os

# 在解压目录中查找 train_convnext.py
target = 'train_convnext.py'
project_dir = None
for root, dirs, files in os.walk('/content/cifar100_project'):
    if target in files:
        project_dir = root
        break

if project_dir:
    os.chdir(project_dir)
    print(f'工作目录: {os.getcwd()}')
    !python train_convnext.py --config config_colab.yaml --drive-sync /content/drive/MyDrive/cifar100_checkpoints
else:
    print(f'未找到 {target}，请检查解压结果')

In [ ]:
# 确认 Drive 同步状态 + 断点续训命令
import os

drive_dir = '/content/drive/MyDrive/cifar100_checkpoints'
if os.path.isdir(drive_dir):
    files = sorted([f for f in os.listdir(drive_dir) if f.endswith('.pth')])
    print(f'Google Drive 中的 checkpoint ({len(files)} 个):')
    for f in files:
        print(f'  {f}')
else:
    print('Drive 同步目录不存在')

print('\n断点续训命令（Colab 断开后重新连接时使用）:')
print(f'  !python train_convnext.py --config config_colab.yaml --resume {drive_dir}/cifar100_best.pth --drive-sync {drive_dir}')